# PitchGraph — build the data lake on Google Drive

Streams all StatsBomb Open Data (3,961 matches, 80 competition-seasons) into compact Parquet tables in your Google Drive. Raw JSON is never saved. Expect **~1.7 GB** on Drive and roughly an hour for everything; it is **resumable**, so if Colab disconnects, just run it again.

Nothing is stored on your laptop: the code comes from GitHub and the data goes to Drive.

> Data: StatsBomb Open Data. Credit StatsBomb (and use their logo) in anything published.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
REPO=https://github.com/akabra13/football_graph_predictor.git
if [ -d /content/pitchgraph ]; then git -C /content/pitchgraph pull -q; else git clone -q $REPO /content/pitchgraph; fi
pip install -q -e /content/pitchgraph
git -C /content/pitchgraph log --oneline -1

In [ ]:
import os
os.environ['PITCHGRAPH_DATA'] = '/content/drive/MyDrive/pitchgraph'

from pitchgraph.config import lake_dir
from pitchgraph.data.ingest import ingest_all
print('lake:', lake_dir())

## 1. Trial run: one competition
World Cup 2022 (64 matches, includes 360). Check the size and timing before running everything.

In [ ]:
m = ingest_all(only=['43_106'])
m['43_106']

## 2. Everything
Skips anything already done. Re-run this cell if the session drops.

In [ ]:
manifest = ingest_all()

## 3. Sanity check

In [ ]:
import polars as pl
lake = lake_dir()
matches = pl.read_parquet(lake / 'matches' / '*.parquet')
events = pl.scan_parquet(lake / 'events' / '*.parquet')
print('matches:', matches.height, '| with 360:', matches['has_360'].sum())
print('events :', events.select(pl.len()).collect().item())
size = sum(p.stat().st_size for p in lake.rglob('*.parquet'))
print('lake size: %.2f GB' % (size / 1e9))
(matches.group_by(['competition', 'season']).len()
        .sort('len', descending=True).head(15))